<a href="https://colab.research.google.com/github/Arceus2006/AIML-project/blob/main/Greedy%2C_BFS_and_DFS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from collections import defaultdict, deque
import time

# ============================================================
# 1. LOAD DATASET
# ============================================================

file_path = "US Airline Flight Routes and Fares 1993-2024.csv"

df = pd.read_csv(
    file_path,
    engine="python",
    on_bad_lines="skip"
)

print("=" * 60)
print("US AIRLINE FLIGHT ROUTES AND FARES")
print("=" * 60)

print("\nDataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. CREATE GRAPH
# ============================================================

routes = df.dropna(
    subset=["city1", "city2", "nsmiles"]
).copy()

routes["nsmiles"] = pd.to_numeric(
    routes["nsmiles"],
    errors="coerce"
)

routes = routes.dropna(subset=["nsmiles"])

routes = routes[routes["nsmiles"] > 0]

graph = defaultdict(list)

for _, row in routes.iterrows():

    city1 = str(row["city1"]).strip()
    city2 = str(row["city2"]).strip()
    distance = float(row["nsmiles"])

    graph[city1].append((city2, distance))
    graph[city2].append((city1, distance))


print("\nNumber of cities:", len(graph))
print("Number of routes:", len(routes))


# ============================================================
# 3. SHOW AVAILABLE CITIES
# ============================================================

cities = sorted(graph.keys())

print("\nSample cities:")

for i, city in enumerate(cities[:30], 1):
    print(i, ".", city)


# ============================================================
# 4. SELECT START AND GOAL
# ============================================================

# Automatically select two cities
# You can change these later

start = cities[0]
goal = cities[-1]

print("\nStart city:", start)
print("Goal city:", goal)


# ============================================================
# 5. GREEDY SEARCH
# ============================================================

def greedy_search(graph, start, goal):

    current = start
    path = [current]

    visited = {current}

    total_distance = 0

    nodes_explored = 0

    while current != goal:

        nodes_explored += 1

        neighbors = [
            (city, distance)
            for city, distance in graph[current]
            if city not in visited
        ]

        if not neighbors:
            return None, None, nodes_explored

        # Greedy choice:
        # choose closest neighboring city
        next_city, distance = min(
            neighbors,
            key=lambda x: x[1]
        )

        path.append(next_city)

        total_distance += distance

        visited.add(next_city)

        current = next_city

    nodes_explored += 1

    return path, total_distance, nodes_explored


# ============================================================
# 6. BFS
# ============================================================

def bfs(graph, start, goal):

    queue = deque([[start]])

    visited = {start}

    nodes_explored = 0

    while queue:

        path = queue.popleft()

        current = path[-1]

        nodes_explored += 1

        if current == goal:
            return path, nodes_explored

        for neighbor, distance in graph[current]:

            if neighbor not in visited:

                visited.add(neighbor)

                new_path = path + [neighbor]

                queue.append(new_path)

    return None, nodes_explored


# ============================================================
# 7. DFS
# ============================================================

def dfs(graph, start, goal):

    stack = [[start]]

    visited = set()

    nodes_explored = 0

    while stack:

        path = stack.pop()

        current = path[-1]

        if current in visited:
            continue

        visited.add(current)

        nodes_explored += 1

        if current == goal:
            return path, nodes_explored

        for neighbor, distance in graph[current]:

            if neighbor not in visited:

                new_path = path + [neighbor]

                stack.append(new_path)

    return None, nodes_explored


# ============================================================
# 8. FUNCTION TO CALCULATE ROUTE DISTANCE
# ============================================================

def calculate_distance(graph, path):

    if path is None:
        return None

    total = 0

    for i in range(len(path) - 1):

        current = path[i]
        next_city = path[i + 1]

        for neighbor, distance in graph[current]:

            if neighbor == next_city:

                total += distance
                break

    return total


# ============================================================
# 9. GREEDY
# ============================================================

print("\n" + "=" * 60)
print("GREEDY SEARCH")
print("=" * 60)

start_time = time.perf_counter()

greedy_path, greedy_distance, greedy_nodes = greedy_search(
    graph,
    start,
    goal
)

greedy_time = time.perf_counter() - start_time

if greedy_path:

    print("\nRoute:")
    print(" -> ".join(greedy_path))

    print("\nConnections:",
          len(greedy_path) - 1)

    print("Distance:",
          round(greedy_distance, 2),
          "miles")

    print("Nodes explored:",
          greedy_nodes)

    print("Execution time:",
          round(greedy_time, 6),
          "seconds")

else:

    print("No route found.")


# ============================================================
# 10. BFS
# ============================================================

print("\n" + "=" * 60)
print("BREADTH-FIRST SEARCH (BFS)")
print("=" * 60)

start_time = time.perf_counter()

bfs_path, bfs_nodes = bfs(
    graph,
    start,
    goal
)

bfs_time = time.perf_counter() - start_time

bfs_distance = calculate_distance(
    graph,
    bfs_path
)

if bfs_path:

    print("\nRoute:")
    print(" -> ".join(bfs_path))

    print("\nConnections:",
          len(bfs_path) - 1)

    print("Distance:",
          round(bfs_distance, 2),
          "miles")

    print("Nodes explored:",
          bfs_nodes)

    print("Execution time:",
          round(bfs_time, 6),
          "seconds")

else:

    print("No route found.")


# ============================================================
# 11. DFS
# ============================================================

print("\n" + "=" * 60)
print("DEPTH-FIRST SEARCH (DFS)")
print("=" * 60)

start_time = time.perf_counter()

dfs_path, dfs_nodes = dfs(
    graph,
    start,
    goal
)

dfs_time = time.perf_counter() - start_time

dfs_distance = calculate_distance(
    graph,
    dfs_path
)

if dfs_path:

    print("\nRoute:")
    print(" -> ".join(dfs_path))

    print("\nConnections:",
          len(dfs_path) - 1)

    print("Distance:",
          round(dfs_distance, 2),
          "miles")

    print("Nodes explored:",
          dfs_nodes)

    print("Execution time:",
          round(dfs_time, 6),
          "seconds")

else:

    print("No route found.")


# ============================================================
# 12. COMPARISON TABLE
# ============================================================

print("\n" + "=" * 60)
print("COMPARISON")
print("=" * 60)

results = []

if greedy_path:
    results.append([
        "Greedy",
        len(greedy_path) - 1,
        round(greedy_distance, 2),
        greedy_nodes,
        round(greedy_time, 6)
    ])

if bfs_path:
    results.append([
        "BFS",
        len(bfs_path) - 1,
        round(bfs_distance, 2),
        bfs_nodes,
        round(bfs_time, 6)
    ])

if dfs_path:
    results.append([
        "DFS",
        len(dfs_path) - 1,
        round(dfs_distance, 2),
        dfs_nodes,
        round(dfs_time, 6)
    ])


comparison = pd.DataFrame(
    results,
    columns=[
        "Algorithm",
        "Connections",
        "Distance (miles)",
        "Nodes Explored",
        "Execution Time"
    ]
)

print("\n")
print(comparison.to_string(index=False))


# ============================================================
# 13. FINAL RESULT
# ============================================================

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print("""
Greedy Search:
Chooses the closest available city at every step.

BFS:
Finds a route with the minimum number of connections.

DFS:
Explores a route deeply before backtracking.

""")

US AIRLINE FLIGHT ROUTES AND FARES

Dataset shape: (245955, 23)

Columns:
['tbl', 'Year', 'quarter', 'citymarketid_1', 'citymarketid_2', 'city1', 'city2', 'airportid_1', 'airportid_2', 'airport_1', 'airport_2', 'nsmiles', 'passengers', 'fare', 'carrier_lg', 'large_ms', 'fare_lg', 'carrier_low', 'lf_ms', 'fare_low', 'Geocoded_City1', 'Geocoded_City2', 'tbl1apk']

Number of cities: 171
Number of routes: 245955

Sample cities:
1 . Albany, NY
2 . Albuquerque, NM
3 . Allentown/Bethlehem/Easton, PA
4 . Amarillo, TX
5 . Appleton, WI
6 . Asheville, NC
7 . Ashland, WV
8 . Aspen, CO
9 . Atlanta, GA (Metropolitan Area)
10 . Atlantic City, NJ
11 . Augusta, GA
12 . Austin, TX
13 . Bangor, ME
14 . Baton Rouge, LA
15 . Belleville, IL
16 . Bellingham, WA
17 . Bend/Redmond, OR
18 . Billings, MT
19 . Birmingham, AL
20 . Bismarck/Mandan, ND
21 . Bloomington/Normal, IL
22 . Boise, ID
23 . Boston, MA (Metropolitan Area)
24 . Bozeman, MT
25 . Buffalo, NY
26 . Bullhead City, AZ
27 . Burlington, VT
28 . Cedar